# 16.7 - Security

Status: VERIFIED

## What Are We Solving?

ML APIs handle sensitive data and expensive compute. Without security, anyone can abuse your endpoint, steal data, or inject malicious inputs. This unit covers authentication, rate limiting, input validation, and secret management.

## Mental Model

Security is like airport security: authentication is your passport, rate limiting is the queue capacity, input validation is the X-ray scanner, and secrets management is the locked briefcase.

## Simple Rate Limiter

In [1]:
import matplotlib
matplotlib.use('Agg')
import time

class RateLimiter:
    """Token bucket rate limiter."""
    def __init__(self, max_tokens, refill_rate):
        self.max_tokens = max_tokens
        self.refill_rate = refill_rate  # tokens per second
        self.tokens = max_tokens
        self.last_refill = time.time()

    def _refill(self):
        now = time.time()
        elapsed = now - self.last_refill
        self.tokens = min(self.max_tokens, self.tokens + elapsed * self.refill_rate)
        self.last_refill = now

    def allow(self):
        self._refill()
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

# Test the rate limiter
limiter = RateLimiter(max_tokens=5, refill_rate=1.0)
print("Rate Limiter Test (5 tokens, 1/s refill):")
for i in range(8):
    allowed = limiter.allow()
    print(f"  Request {i+1}: {'ALLOWED' if allowed else 'REJECTED'} (tokens left: {limiter.tokens:.1f})")


Rate Limiter Test (5 tokens, 1/s refill):
  Request 1: ALLOWED (tokens left: 4.0)
  Request 2: ALLOWED (tokens left: 3.0)
  Request 3: ALLOWED (tokens left: 2.0)
  Request 4: ALLOWED (tokens left: 1.0)
  Request 5: ALLOWED (tokens left: 0.0)
  Request 6: REJECTED (tokens left: 0.0)
  Request 7: REJECTED (tokens left: 0.0)
  Request 8: REJECTED (tokens left: 0.0)


## Input Validation

In [2]:
import matplotlib
matplotlib.use('Agg')
import re
import numpy as np

class InputValidator:
    """Validate ML prediction inputs."""
    def __init__(self, n_features, max_value=1e6, allowed_dtypes=None):
        self.n_features = n_features
        self.max_value = max_value
        self.allowed_dtypes = allowed_dtypes or [float, int, np.floating, np.integer]

    def validate(self, features):
        errors = []
        if not isinstance(features, (list, np.ndarray)):
            errors.append("features must be a list or array")
            return errors
        if len(features) != self.n_features:
            errors.append(f"expected {self.n_features} features, got {len(features)}")
        for i, val in enumerate(features):
            if not isinstance(val, tuple(self.allowed_dtypes)):
                errors.append(f"feature[{i}]: invalid type {type(val).__name__}")
            elif abs(float(val)) > self.max_value:
                errors.append(f"feature[{i}]: value {val} exceeds max {self.max_value}")
        return errors

validator = InputValidator(n_features=3)
test_cases = [
    ('valid', [1.0, 2.0, 3.0]),
    ('wrong count', [1.0, 2.0]),
    ('bad type', [1.0, 'two', 3.0]),
    ('overflow', [1.0, 2.0, 9999999.0]),
]
print("Input Validation Results:")
for name, features in test_cases:
    errors = validator.validate(features)
    status = 'VALID' if not errors else 'INVALID'
    print(f"  {name:15s}: {status}")
    for e in errors:
        print(f"    -> {e}")


Input Validation Results:
  valid          : VALID
  wrong count    : INVALID
    -> expected 3 features, got 2
  bad type       : INVALID
    -> feature[1]: invalid type str
  overflow       : INVALID
    -> feature[2]: value 9999999.0 exceeds max 1000000.0


## Security Checklist for ML APIs

- [ ] **API keys** or OAuth for authentication
- [ ] **Rate limiting** per client/IP
- [ ] **Input validation** on every endpoint
- [ ] **Secrets in env vars**, never in code
- [ ] **HTTPS only** — no plaintext in production
- [ ] **CORS** restricted to known origins
- [ ] **Dependency scanning** (safety, trivy)
- [ ] **Model artifact integrity** — checksum verification

In [3]:
import matplotlib
matplotlib.use('Agg')
import hashlib

# Secret management pattern
def mask_secret(value, show_last=4):
    if len(value) <= show_last:
        return '*' * len(value)
    return '*' * (len(value) - show_last) + value[-show_last:]

secrets = {
    'DATABASE_URL': 'postgresql://admin:s3cret_p@ss@db:5432/ml',
    'API_KEY': 'sk-abcdef1234567890abcdef',
    'MODEL_HASH': hashlib.sha256(b'model_artifact').hexdigest(),
}

print("Secret Management:")
for key, value in secrets.items():
    print(f"  {key:15s}: {mask_secret(value)}")
print("\nAlways use environment variables for secrets.")
print("Never log full secrets — use mask_secret() in logs.")


Secret Management:
  DATABASE_URL   : *************************************2/ml
  API_KEY        : *********************cdef
  MODEL_HASH     : ************************************************************d720

Always use environment variables for secrets.
Never log full secrets — use mask_secret() in logs.


In [4]:
import matplotlib
matplotlib.use('Agg')
print('VERIFICATION PASSED: Phase 16.7 complete')


VERIFICATION PASSED: Phase 16.7 complete